# Quantum Teleportation and beyond

For this notebook, I used the exercises from the course of Prof. Jens Eisert on QIT - WS23/24, available online.

In [ ]:
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, entropy, state_fidelity, random_statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from IPython.display import display
import matplotlib.pyplot as plt
import sympy
import numpy as np

The basic layout of Quantum Teleportation is as follows. Alice has an unknown qubit  (called qubit 0) $$ \ket{\psi}=\alpha\ket{0}+\beta\ket{1} $$
Alice and Bob pre-share a Bell pair (qubit 1 = Alice's half, qubit 2 = Bob's half). Alice wants Bob to end up holding $
\ket{\psi} $ using only a classical channel - meaning, Alice will share two classical bits to Bob. The basic circuit is the following.

Note that Alice will measure in the Bell basis. She does this by first rotating from the Bell basis into the computational basis and then measure. $ CNOT \otimes H $ is equivalent to the inverse of Bell pair creation, in other words, a perfect bijection from the 4 Bell states to the 4 computational basis states.

Depending on the outcome $(ab)$ where $a,b=0,1$, Bob will apply certain operations to recover $\ket{\psi}$

In [ ]:
# We name the registers for ease of reading the diagram
psi = QuantumRegister(1, 'ψ')        # Alice's unknown qubit state
alice = QuantumRegister(1, 'Alice')   # Alice's half of Bell pair
bob = QuantumRegister(1, 'Bob')       # Bob's half of Bell pair
crz = ClassicalRegister(1, r'$c_0$')     # Alice's first classical bit
crx = ClassicalRegister(1, r'$c_1$')     # Alice's second classical bit

qc = QuantumCircuit(psi, alice, bob, crz, crx)

# Bell pair creation that is shared
qc.h(alice)
qc.cx(alice, bob)
qc.barrier()

# Alice measures in the Bell basis
qc.cx(psi, alice)
qc.h(psi)
qc.barrier()
qc.measure(psi, crz)
qc.measure(alice, crx)
qc.barrier()

# Bob's corrections
with qc.if_test((crx, 1)):
    qc.x(bob)
with qc.if_test((crz, 1)):
    qc.z(bob)

display(qc.draw('mpl'))

We build a function that performs the teleportation for an arbitrary state $\ket{\psi}$ and then measure with a simulator. We verify that the state was teleported by computing the fidelity between the initial statevector and Bob's final qubit.

In [ ]:
def teleport(psi:Statevector) -> QuantumCircuit:
    qc = QuantumCircuit(3, 2)
    qc.initialize(psi, 0)
    qc.h(1)
    qc.cx(1, 2)
    # Alice Bell Basis measurement
    qc.cx(0, 1)  # control = unknown state, target = Alice's Bell qubit
    qc.h(0)
    qc.measure([0, 1], [0, 1])  # measure qubit 0->cbit 0, qubit 1->cbit 1
    # Alice shares classical bits
    # Bob corrections on qubit 2 based on Alice's classical bits:
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(2)  # if cbit 1 == 1, apply X on Bob's qubit
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)  # if cbit 0 == 1, apply Z on Bob's qubit

    qc.save_statevector()
    fig = qc.draw(output = 'mpl')
    display(fig)
    return qc

sv = random_statevector(2)
qc = teleport(sv)

simulator = AerSimulator(method="statevector")

result = simulator.run(qc).result()

final_state = result.get_statevector()
# Trace out Alice's qubits (0 and 1)
bob_state = partial_trace(final_state, [0, 1])

fidelity = state_fidelity(bob_state, sv)

print(f"Fidelity: {fidelity}")

Note that any Bell state can be transformed into another Bell state by local operations only. To see this, it's sufficient to show it for $\ket{\Phi^+}$.


\begin{aligned}
\ket{\Phi^-}
&= (\mathbb{I} \otimes Z)\ket{\Phi^+}
= (Z \otimes \mathbb{I})\ket{\Phi^+} \\[4pt]
\ket{\Psi^+}
&= (\mathbb{I} \otimes X)\ket{\Phi^+}
= (X \otimes \mathbb{I})\ket{\Phi^+} \\[4pt]
\ket{\Psi^-}
&= (\mathbb{I} \otimes Y)\ket{\Phi^+}
= (Y \otimes \mathbb{I})\ket{\Phi^+},
\end{aligned}

### A variant
Actually, classical bits in standard teleportation are not always necessary and when the state space is restricted, we can do better. Suppose that Alice wants to transmit the following state to Bob,
$$ \ket{\alpha} = \frac{1}{\sqrt{2}} \left ( \ket{0} + e^{-i\alpha} \ket{1} \right )$$
for $\alpha \in (0,2\pi)$. This is a special state because it lives on the surface of the Bloch sphere. Assuming further that Alice knows the value of $\alpha$. 

Instead of the $\{\ket{0}, \ket{1}\}$ basis, we may use the one $\{\ket{\alpha},\ket{\alpha^{\perp}}\}$ for $\ket{\alpha^{\perp}} \equiv \ket{\alpha + \pi}$. 

The teleportation protocol is then as follows: Alice and Bob pre-share the $\ket{\Phi^+}$ bell state. Alice measures $\ket{\Phi^+}$ in the $\{\ket{\alpha},\ket{\alpha^{\perp}}\}$ basis. Then Bob can fully produce $\ket{\alpha}$ with only a single bit of information be shared.

To write the protocol with Qiskit we'll use a unitary gate that implements the rotation into the different basis.

In [ ]:
from qiskit.circuit.library import UnitaryGate

def U_alpha_dag(alpha):
    # U_alpha maps |0> -> |alpha>, |1> -> |alpha_perp>
    U = np.array([
        [1/np.sqrt(2), 1/np.sqrt(2)],
        [np.exp(-1j*alpha)/np.sqrt(2), -np.exp(-1j*alpha)/np.sqrt(2)]
    ]).conj().T  # take the conjugate transpose = dagger
    return UnitaryGate(U, label=r'$U^{\dag}_\alpha$')

def teleport(alpha:float) -> QuantumCircuit:
    qc = QuantumCircuit(3, 1) # Note a single classical bit is needed
    gate = U_alpha_dag(alpha)

    # qubit 0 is the alpha state
    state_alpha = [1/np.sqrt(2), 1/np.sqrt(2)*np.exp(-1j*alpha)] 
    qc.initialize(state_alpha, 0)

    # Bell pair between qubit 1 and 2
    qc.h(1)
    qc.cx(1, 2)

    # Alice Bell Basis measurement
    qc.append(gate, [0])
    qc.measure(0, 0)  # measure qubit 0->cbit 0

    # Alice shares classical bits
    
    # Bob's corrections
    qc.x(2)  # always apply X
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)
    qc.save_statevector()  # capture AFTER corrections
    fig = qc.draw(output = 'mpl')
    display(fig)
    return qc

alpha = np.pi/4
qc = teleport(alpha)

simulator = AerSimulator(method="statevector")
result = simulator.run(qc, shots=1).result()

# Check measurement outcome
counts = result.get_counts()
print(f"Measurement counts: {counts}")

# Check statevector
final_state = result.get_statevector()
bob_state = partial_trace(final_state, [0, 1])
print(f"Bob's reduced state:\n{bob_state}")

sv = Statevector([1/np.sqrt(2), 1/np.sqrt(2) * np.exp(-1j * alpha)])
print(f"Target state: {sv.data}")
print(f"Fidelity: {state_fidelity(bob_state, sv):.6f}")